<a href="https://colab.research.google.com/github/linapioche/nlp-text-summarization/blob/main/01_exploration_samsum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch
print(torch.cuda.is_available())


True


In [7]:
!pip install datasets

In [8]:
from datasets import load_dataset

In [9]:
dataset_samsum=load_dataset("knkarthick/samsum")

README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [10]:
print(len(dataset_samsum["test"]))

819


In [11]:
print(len(dataset_samsum["train"]))

14731


In [12]:
print(len(dataset_samsum["validation"]))

818


Après avoir chargé le dataset SAMSum et vérifié la taille de ses trois parties,nous observons quelques exemples au hasard afin de comprendre la nature des données (style des dialogues, forme des résumés) avant de décider des étapes de prétraitement.

In [13]:
import random

random.seed(42)  # mêmes exemples à chaque exécution

n = len(dataset_samsum["train"])        # nombre de lignes du train
indices = random.sample(range(n), 5)

for i in indices:
    ligne = dataset_samsum["train"][i]  # un dictionnaire : id, dialogue, summary
    print("DIALOGUE :")
    print(ligne["dialogue"])
    print("RÉSUMÉ :")
    print(ligne["summary"])
    print("-" * 50)



DIALOGUE :
David: The new movie of Jonhy English has come out, have you seen it?
Patricia: No but I have been meaning to go tough. I heard it's hilarious.
David: Rowan Atkison is just awesome, love that guy! In Mr. Bean I would just laugh so hard ahaha
Patricia: Me too 😂 I couldn't watch some scenes sometimes cause they would make me nervous from all the constant crap he did ahhaha
David: ahahaa xD  Anyway.. wanna go to the 21:40 session today? I ain't got much going on so..
Patricia: Sure! Where are you having dinner?
David: Was thinking of just ordering a pizza, you have any ideas?
Patricia: There's a new Mexican place and they do take out's, want me to grab something and meet you at your place?
David: Oh that's what I'm talking about! Bring me 2 chicken burritos and nachoooos with guacamole.
Patricia: Anything else for the little boy? ahaha xD
David: While you're at it a coke would do 😂
Patricia: Jesus.. x) Leaving my place now, cya in a bit.
RÉSUMÉ :
Patricia and David are going to

Ces exemples montrent des dialogues informels (abréviations, fautes, émojis, balises comme <file_video>) de longueurs très variables, résumés à la troisième personne. Nous allons maintenant mesurer les longueurs des dialogues et des résumés pour choisir les paramètres de troncature, puis construire le    prétraitement et le vocabulaire

In [14]:
import numpy as np

# Lignes vides ou manquantes, sur les 3 parties
for nom_partie in ["train", "validation", "test"]:
    partie = dataset_samsum[nom_partie]
    vides_dial = sum(1 for t in partie["dialogue"] if t is None or t.strip() == "")
    vides_resume = sum(1 for t in partie["summary"] if t is None or t.strip() == "")
    print(nom_partie, "| dialogues vides :", vides_dial, "| résumés vides :", vides_resume)

# Longueurs en nombre de mots (sur le train)
train = dataset_samsum["train"]
long_dial = [len(t.split()) for t in train["dialogue"]]
long_resume = [len(t.split()) for t in train["summary"]]

for nom, longueurs in [("dialogues", long_dial), ("résumés", long_resume)]:
    print(nom,
          "| moyenne :", round(np.mean(longueurs), 1),
          "| médiane :", np.median(longueurs),
          "| max :", np.max(longueurs),
          "| 95e percentile :", np.percentile(longueurs, 95))

train | dialogues vides : 0 | résumés vides : 0
validation | dialogues vides : 0 | résumés vides : 0
test | dialogues vides : 0 | résumés vides : 0
dialogues | moyenne : 93.8 | médiane : 73.0 | max : 803 | 95e percentile : 237.0
résumés | moyenne : 20.3 | médiane : 18.0 | max : 64 | 95e percentile : 43.0


**Vérification des données et longueurs des textes**

Nous avons vérifié que les trois parties (train : 14 731, validation : 818, test : 819) ne contiennent aucun dialogue ni résumé vide. Nous avons ensuite mesuré la longueur des textes en nombre de mots sur le train : les dialogues font en moyenne 93,8 mots (médiane 73, maximum 803, 95e percentile 237) et les résumés 20,3 mots (médiane 18, maximum 64, 95e percentile 43).

Les dialogues sont donc environ 4 à 5 fois plus longs que les résumés, et les très longs textes sont rares. Les textes sont tronqués au 95e percentile de leur longueur (240 mots pour les dialogues, 45 pour les résumés) : le modèle a besoin de séquences de taille fixe, et ce seuil garde 95 % des textes entiers sans gaspiller de calcul sur de rares dialogues très longs.

In [15]:
import re

# Le "motif" décrit ce qu'on veut extraire du texte. Trois possibilités, séparées par "|" :
#   1) <[a-z_0-9]+>        -> une balise entière, comme <file_video>
#   2) \w+(?:['’]\w+)?     -> un mot, avec son apostrophe éventuelle (don't, i'm)
#   3) [^\w\s]             -> un seul signe de ponctuation (. , ? ! : 😂 ...)
MOTIF = re.compile(r"<[a-z_0-9]+>|\w+(?:['’]\w+)?|[^\w\s]")

def tokenize(texte):
    texte = texte.lower()          # décision 1 : tout en minuscules
    return MOTIF.findall(texte)    # on récupère tous les morceaux qui correspondent au motif

# Test sur une phrase inspirée de tes exemples
print(tokenize("Patricia: Me too 😂 I couldn't watch <file_video>, ok?"))

['patricia', ':', 'me', 'too', '😂', 'i', "couldn't", 'watch', '<file_video>', ',', 'ok', '?']


In [16]:
#test suivant
tokenize(dataset_samsum["train"][0]["dialogue"])

['amanda',
 ':',
 'i',
 'baked',
 'cookies',
 '.',
 'do',
 'you',
 'want',
 'some',
 '?',
 'jerry',
 ':',
 'sure',
 '!',
 'amanda',
 ':',
 "i'll",
 'bring',
 'you',
 'tomorrow',
 ':',
 '-',
 ')']

La fonction de tokenisation passe le texte en minuscules, conserve les balises (<file_video>) et les mots avec apostrophe (i'll) comme tokens uniques, et sépare chaque signe de ponctuation. Les émoticônes textuelles comme :-) sont découpées en plusieurs tokens, ce qui constitue une limite mineure.